# Full Pipeline Runner (Notebook)

This notebook runs the full CLI pipeline and then visualizes training results.

Pipeline stages:
1. Preprocessing
2. Keyframe extraction
3. Landmark extraction
4. Training

Split note:
- Current project training uses **train/validation split only** (default 80/20).
- There is **no separate test split** in the current trainer script by default.
- This notebook includes an optional holdout-test evaluation cell.

In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import numpy as np
import matplotlib.pyplot as plt

project_root = Path.cwd()
if not (project_root / 'pyproject.toml').exists():
    # If notebook opens from a different cwd, force project root using this file path.
    project_root = Path('H:/Projects/sem-5-ml-project')

pipeline_script = project_root / 'util_scripts' / 'run_full_pipeline.py'
trainer_script = project_root / 'model' / 'trainer_current_config(gpu).py'

dataset_dir = project_root / 'dataset' / 'raw_video_data'
preprocessed_dir = project_root / 'outputs' / 'preprocessed'
keyframes_dir = project_root / 'outputs' / 'keyframes'
landmarks_dir = project_root / 'outputs' / 'landmarks'
checkpoint_dir = project_root / 'checkpoints' / 'rtx2060_full_run'

print('Project root    :', project_root)
print('Pipeline script :', pipeline_script)
print('Trainer script  :', trainer_script)
print('Checkpoint dir  :', checkpoint_dir)

## 1) Run Full Pipeline

Set `RUN_PIPELINE = True` to execute all stages.

Tip: For quick checks, add one or more skip flags in `cmd` below, for example `--skip-preprocessing` and `--skip-keyframes`.

In [ ]:
RUN_PIPELINE = False

cmd = [
    sys.executable,
    str(pipeline_script),
    '--dataset-dir', str(dataset_dir.relative_to(project_root)),
    '--preprocessed-dir', str(preprocessed_dir.relative_to(project_root)),
    '--keyframes-dir', str(keyframes_dir.relative_to(project_root)),
    '--landmarks-dir', str(landmarks_dir.relative_to(project_root)),
    '--checkpoint-dir', str(checkpoint_dir.relative_to(project_root)),
    '--trainer-script', str(trainer_script.relative_to(project_root)),
    '--model-type', 'lstm',
    '--batch-size', '8',
    '--num-epochs', '50',
    '--device', 'cuda',
    '--num-workers', '4',
    '--use-mixed-precision',
    '--grad-accum-steps', '2',
]

if RUN_PIPELINE:
    result = subprocess.run(cmd, cwd=project_root, check=False)
    print('Exit code:', result.returncode)
else:
    print('RUN_PIPELINE is False. Set it to True to run training from this notebook.')
    print(' '.join(cmd))

## 2) Confirm Current Split Behavior

Current training loader creates:
- Train split: 80%
- Validation split: 20%
- Test split: not created by default

In [ ]:
from model.dataset import SignLanguageDataset, create_data_loaders

dataset = SignLanguageDataset(landmarks_dir=landmarks_dir)
train_loader, val_loader = create_data_loaders(
    landmarks_dir=landmarks_dir,
    batch_size=8,
    train_split=0.8,
    random_seed=42,
)

print('Total samples:', len(dataset))
print('Train samples:', len(train_loader.dataset))
print('Val samples  :', len(val_loader.dataset))
print('Test samples : 0 (not used by default trainer)')

## 3) Visualize Training Curves

This reads `training_history.json` from your checkpoint directory.

In [ ]:
history_path = checkpoint_dir / 'training_history.json'
if not history_path.exists():
    raise FileNotFoundError(f'Training history not found: {history_path}')

with open(history_path, 'r', encoding='utf-8') as f:
    history = json.load(f)

epochs = range(1, len(history.get('train_loss', [])) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(epochs, history.get('train_loss', []), label='train_loss')
axes[0].plot(epochs, history.get('val_loss', []), label='val_loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, history.get('train_acc', []), label='train_acc')
axes[1].plot(epochs, history.get('val_acc', []), label='val_acc')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(epochs, history.get('learning_rates', []), label='lr')
axes[2].set_title('Learning Rate')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.tight_layout()
plt.show()

## 4) Validation Confusion Matrix and Metrics

Loads `best_model.pth` and evaluates on validation split.

In [ ]:
import torch
from sklearn.metrics import confusion_matrix, classification_report
from model.sign_classifier import create_model

checkpoint_path = checkpoint_dir / 'best_model.pth'
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Best checkpoint not found: {checkpoint_path}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint = torch.load(checkpoint_path, map_location=device)
cfg = checkpoint.get('config', {})

model_type = cfg.get('model_type', 'lstm')
num_classes = int(cfg.get('num_classes', len(dataset.classes)))
landmark_dim = int(cfg.get('landmark_dim', dataset[0][0].shape[-1]))

model = create_model(
    model_type=model_type,
    num_classes=num_classes,
    device=device,
    landmark_dim=landmark_dim,
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds = []
all_labels = []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(y.cpu().numpy().tolist())

labels_sorted = dataset.classes
cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(labels_sorted))))

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest')
plt.title('Validation Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(len(labels_sorted))
plt.xticks(tick_marks, labels_sorted, rotation=45, ha='right')
plt.yticks(tick_marks, labels_sorted)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

print(classification_report(all_labels, all_preds, target_names=labels_sorted, zero_division=0))

## 5) Optional Test Split Evaluation

Use this if you want a separate test holdout right now without changing project trainer code.

In [ ]:
from torch.utils.data import random_split, DataLoader

RUN_TEST_EVAL = False
if RUN_TEST_EVAL:
    full_dataset = SignLanguageDataset(landmarks_dir=landmarks_dir)
    n = len(full_dataset)
    n_train = max(1, int(0.7 * n))
    n_val = max(1, int(0.15 * n))
    n_test = n - n_train - n_val
    if n_test < 1:
        n_test = 1
        n_val = max(1, n_val - 1)

    gen = torch.Generator().manual_seed(42)
    train_ds, val_ds, test_ds = random_split(full_dataset, [n_train, n_val, n_test], generator=gen)

    test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)

    test_preds, test_labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            preds = torch.argmax(logits, dim=1)
            test_preds.extend(preds.cpu().numpy().tolist())
            test_labels.extend(y.cpu().numpy().tolist())

    print('Split sizes -> train:', n_train, 'val:', n_val, 'test:', n_test)
    print(classification_report(test_labels, test_preds, target_names=full_dataset.classes, zero_division=0))
else:
    print('RUN_TEST_EVAL is False. Set it to True to run holdout test evaluation.')